[![../img/pythonista.png](../img/pythonista.png)](https://www.pythonista.io)

# *Middleware*: controlar el ciclo del agente.

**py501 · Aplicaciones Agénticas con LangChain** · Pythonista® · Instructor: José Luis Chiquete · Módulo 5: *Middleware*

> 🔑 **Versión con soluciones.** Solo para el instructor o para revisión posterior.

En este *notebook* se agregará control y gobierno al agente **sin reescribirlo**: aprobación humana, redacción de datos personales, límites de llamadas y *middleware* propio con decoradores.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* **Ejercicio básico:** lo que todos deben dominar.
* **Reto (opcional):** para quien quiere profundizar. Se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.
* Cada ejecución llama a *Gemini* (el modelo del taller): las respuestas varían entre ejecuciones y consumen la cuota de tu `GOOGLE_API_KEY`.

### Preparación del entorno.

* La siguiente celda localizará `data/`, cargará `.env` si existe (solo en ejecución local), definirá el modelo del taller (*Gemini*) y verificará que exista la clave `GOOGLE_API_KEY`. Si falla, sigue el *notebook* `00_claves_de_api_y_secretos.ipynb`.

In [ ]:
import os

from dotenv import load_dotenv

RAIZ = next(p for p in (os.curdir, os.pardir)
            if os.path.isdir(os.path.join(p, "data", "politicas")))
DATA = os.path.join(RAIZ, "data")
POLITICAS = os.path.join(DATA, "politicas")
load_dotenv(os.path.join(RAIZ, ".env"))   # solo tiene efecto en ejecución local; en Codespaces se usan los secretos

# Este taller usa Gemini. Cualquier proveedor funciona cambiando la cadena "proveedor:modelo" (ver notebook 03).
MODELO = os.environ.get("MODELO", "google_genai:gemini-3.8-flash")
EMBEDDINGS = os.environ.get("EMBEDDINGS", "google_genai:gemini-embedding-001")

if MODELO.startswith("google_genai") and not (os.environ.get("GOOGLE_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    raise RuntimeError("Falta GOOGLE_API_KEY. Sigue el notebook 00_claves_de_api_y_secretos: guárdala como secreto de Codespaces "
                       "con acceso a este repositorio y vuelve a abrir el Codespace.")
MODELO

* La siguiente celda importará las bibliotecas de este *notebook*.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import (HumanInTheLoopMiddleware, ModelCallLimitMiddleware,
                                         PIIMiddleware, ToolCallLimitMiddleware, wrap_tool_call)
from langchain_core.messages import ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command, interrupt

* La siguiente celda definirá las herramientas de políticas (objetos de los *notebooks* anteriores).

In [ ]:
from langchain.tools import tool


@tool
def listar_politicas() -> list[str]:
    """Lista los nombres de las políticas internas disponibles."""
    return sorted(f[:-3] for f in os.listdir(POLITICAS) if f.endswith(".md"))


@tool
def leer_politica(nombre: str) -> str:
    """Devuelve el texto completo de una política interna.

    Args:
        nombre: nombre exacto devuelto por listar_politicas, sin extensión.
    """
    ruta = os.path.join(POLITICAS, f"{nombre}.md")
    if not os.path.isfile(ruta):
        return f"No existe la política '{nombre}'. Disponibles: {', '.join(listar_politicas.invoke({}))}."
    with open(ruta, encoding="utf-8") as archivo:
        return archivo.read()

* La siguiente celda conectará el servidor MCP de tickets.

In [ ]:
import sys

from langchain_mcp_adapters.client import MultiServerMCPClient

cliente_tickets = MultiServerMCPClient({
    "tickets": {
        "transport": "stdio",
        "command": sys.executable,
        "args": [os.path.join(RAIZ, "mcp_servers", "tickets_server.py")],
    }
})
herramientas_mcp = await cliente_tickets.get_tools()
[h.name for h in herramientas_mcp]

---
## 1. ¿Qué es el *middleware*?

El *middleware* son ganchos que se ejecutan **alrededor** de cada paso del ciclo del agente: antes y después de llamar al modelo, y alrededor de cada llamada a una herramienta. Permite cambiar el comportamiento sin tocar el agente.

| *Middleware* incluido | Qué hace |
|---|---|
| `HumanInTheLoopMiddleware` | Pausa antes de ciertas herramientas y espera aprobación, edición o rechazo |
| `PIIMiddleware` | Detecta y redacta, enmascara o bloquea datos personales (correos, tarjetas, IP…) |
| `ModelCallLimitMiddleware` / `ToolCallLimitMiddleware` | Cortan el ciclo tras N llamadas |
| `SummarizationMiddleware` | Resume el historial cuando crece demasiado |
| `ModelRetryMiddleware` / `ModelFallbackMiddleware` | Reintentos y modelo de respaldo |
| `LLMToolSelectorMiddleware` | Elige qué herramientas mostrar al modelo en cada paso |

Se ejecutan **en el orden de la lista** al entrar y en orden inverso al salir: el primero de la lista envuelve a todos los demás.

---
## 2. La pila de *middleware*.

* La siguiente celda creará el agente con cuatro *middleware*. La aprobación humana **requiere un *checkpointer***: el agente se pausa, guarda su estado y espera.

In [ ]:
agente = create_agent(
    model=MODELO,
    tools=[listar_politicas, leer_politica, *herramientas_mcp],
    system_prompt=("Eres el asistente de soporte interno. El usuario actual es 'ana.lopez'. "
                   "Lee la política aplicable antes de crear un ticket."),
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
        ModelCallLimitMiddleware(run_limit=8),
        ToolCallLimitMiddleware(run_limit=6),
        HumanInTheLoopMiddleware(interrupt_on={
            "crear_ticket": {"allowed_decisions": ["approve", "edit", "reject"]},
            "cerrar_ticket": True,
        }),
    ],
    checkpointer=InMemorySaver(),
)

* La siguiente celda ejecutará el agente hasta la interrupción y mostrará la acción pendiente.

In [ ]:
config = {"configurable": {"thread_id": "nb08-1"}}
r = await agente.ainvoke({"messages": [{"role": "user", "content":
    "Me robaron la laptop hoy. Escríbanme a ana.lopez@andina.mx"}]}, config)

print("Mensaje que vio el modelo:", r["messages"][0].text)
for solicitud in r["__interrupt__"][0].value["action_requests"]:
    print("Pendiente de aprobación:", solicitud["name"], solicitud["args"])

**Observa:** el correo llegó al modelo como `[REDACTED_EMAIL]`, y el agente quedó **pausado** antes de crear el ticket.

* La siguiente celda reanudará el agente con la decisión humana.

In [ ]:
decision = {"type": "approve"}
# Alternativas:
# decision = {"type": "reject", "message": "No abras ticket: ya lo reportó su jefe."}
# decision = {"type": "edit", "edited_action": {"name": "crear_ticket", "args": {...}}}

r = await agente.ainvoke(Command(resume={"decisions": [decision]}), config)
r["messages"][-1].text

**💡 Ideas clave de la sección 2:**
* El *middleware* agrega control sin reescribir el agente.
* La aprobación humana pausa el grafo y se reanuda con `Command(resume=...)` en el mismo `thread_id`.
* La redacción de datos personales ocurre **antes** de que el texto llegue al modelo.

### Ejercicio 1 · Decisiones distintas.

Repite el flujo con un `thread_id` nuevo, pero esta vez **edita** la acción para cambiar la prioridad a `"normal"` y agregar `"(editado por supervisor)"` al título. Verifica con `consultar_tickets` que el ticket quedó con los datos editados.

**🎯 Objetivo:** usar la decisión `edit`. **Qué se espera aprender:** que la persona puede corregir la acción del agente, no solo aprobarla o rechazarla.

* La siguiente celda muestra una solución posible.

In [ ]:
config_2 = {"configurable": {"thread_id": "nb08-2"}}
r = await agente.ainvoke({"messages": [{"role": "user", "content": "Mi monitor parpadea, ábreme un ticket urgente."}]}, config_2)
pendiente = r["__interrupt__"][0].value["action_requests"][0]
args = {**pendiente["args"], "prioridad": "normal", "titulo": pendiente["args"]["titulo"] + " (editado por supervisor)"}
r = await agente.ainvoke(Command(resume={"decisions": [{"type": "edit", "edited_action": {"name": "crear_ticket", "args": args}}]}), config_2)

consultar = next(h for h in herramientas_mcp if h.name == "consultar_tickets")
await consultar.ainvoke({"usuario": "ana.lopez"})

---
## 3. *Middleware* propio.

Con decoradores como `@wrap_tool_call`, `@wrap_model_call`, `@before_model` o `@dynamic_prompt` se escribe *middleware* en pocas líneas. Como este agente se ejecuta con `ainvoke` (herramientas MCP), el *middleware* se escribe con `async`.

* La siguiente celda definirá un *middleware* que registra cada llamada a herramienta.

In [ ]:
@wrap_tool_call
async def registrar_herramientas(request, handler):
    print(f"→ {request.tool_call['name']}({request.tool_call['args']})")
    return await handler(request)

### Ejercicio 2 · Aprobar solo lo urgente.

Escribe un *middleware* `aprobar_urgentes` con `@wrap_tool_call` que pida aprobación (con `interrupt`) **solo** cuando se llame `crear_ticket` con prioridad `"urgente"`. Si la respuesta no es `"aprobar"`, devuelve un `ToolMessage` que explique el rechazo sin ejecutar la herramienta. Úsalo en lugar de `HumanInTheLoopMiddleware` y pruébalo con un caso urgente y uno normal.

**🎯 Objetivo:** escribir *middleware* con lógica propia. **Qué se espera aprender:** que las reglas de negocio (qué requiere aprobación) viven en el *middleware*, no en el *prompt*.

* La siguiente celda muestra una solución posible.

In [ ]:
@wrap_tool_call
async def aprobar_urgentes(request, handler):
    llamada = request.tool_call
    if llamada["name"] == "crear_ticket" and llamada["args"].get("prioridad") == "urgente":
        decision = interrupt({"accion": "crear_ticket", "args": llamada["args"]})
        if decision != "aprobar":
            return ToolMessage("Un supervisor rechazó crear este ticket urgente.", tool_call_id=llamada["id"])
    return await handler(request)


agente_urgentes = create_agent(MODELO, tools=[listar_politicas, leer_politica, *herramientas_mcp],
    system_prompt="Eres el asistente de soporte interno. El usuario actual es 'ana.lopez'.",
    middleware=[registrar_herramientas, aprobar_urgentes], checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": "nb08-urgente"}}
r = await agente_urgentes.ainvoke({"messages": [{"role": "user", "content": "¡Me robaron la laptop!"}]}, cfg)
print("Interrumpido:", "__interrupt__" in r)
r = await agente_urgentes.ainvoke(Command(resume="aprobar"), cfg)
print(r["messages"][-1].text)

r = await agente_urgentes.ainvoke({"messages": [{"role": "user", "content": "Quiero instalar Tableau, ábreme ticket normal."}]},
                                  {"configurable": {"thread_id": "nb08-normal"}})
print("Interrumpido:", "__interrupt__" in r)

### Reto 1 · Límite bajo.

Crea el agente con `ToolCallLimitMiddleware(run_limit=1)` y hazle una pregunta que requiera leer dos políticas. ¿Cómo termina el agente? Revisa el último mensaje.

**🎯 Objetivo:** observar el efecto de un límite. **Qué se espera aprender:** que un límite protege el costo, pero el agente debe poder explicar que no terminó.

**✅ Respuesta sugerida:**

Tras la primera herramienta, el *middleware* bloquea las siguientes: con `exit_behavior="continue"` (valor por defecto) el agente recibe un mensaje de error en lugar del resultado y suele responder con lo que tiene, a veces incompleto. Con `exit_behavior="end"` el ciclo termina de inmediato. En producción conviene que el agente avise al usuario que no completó la tarea.

### Reto 2 · Orden de la pila.

Caso de diseño: el agente puede **transferir dinero** entre cuentas. Ordena estos *middleware* y justifica: aprobación humana, redacción de datos personales, límite de llamadas a herramientas, registro de auditoría, modelo de respaldo.

**🎯 Objetivo:** razonar sobre el orden de ejecución. **Qué se espera aprender:** que el orden determina qué ve cada capa.

**✅ Respuesta sugerida:**

Una propuesta: 1) **registro de auditoría** (primero, para registrar todo, incluso lo que otras capas bloquean); 2) **redacción de datos personales** (antes de que el texto llegue al modelo o a los registros externos); 3) **límite de llamadas** (corta bucles antes de llegar a acciones costosas); 4) **aprobación humana** sobre la herramienta de transferencia (lo más cerca posible de la acción); 5) **modelo de respaldo** (envuelve solo la llamada al modelo). Lo importante es justificar qué ve cada capa.

---
## ✅ Cierre.

**Práctica:** terminar los ejercicios 1 y 2.

**Siguiente *notebook*:** `09_langgraph_estado_y_grafos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>